In [1]:
import pandas as pd
import numpy as np

In [2]:
riders = pd.read_csv("../dataset/processed/riders_features.csv")

print(riders.shape)

riders.head()

(45493, 33)


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,...,Day_of_Week,Month,Weekend,Peak_Period,Trip_Distance_km,Traffic_Score,Weather_Score,Vehicle_Score,Rider_Experience,Workload
0,0xcdcd,DEHRES17DEL01,36.0,4.2,30.327968,78.046106,30.397968,78.116106,2022-02-12,21:55,...,Saturday,2,1,Dinner,10.280582,4,4,4,151.2,3.0
1,0xd987,KOCRES16DEL01,21.0,4.7,10.003064,76.307589,10.043064,76.347589,2022-02-13,14:55,...,Sunday,2,1,Lunch,6.242319,3,5,4,98.7,1.0
2,0x2784,PUNERES13DEL03,23.0,4.7,18.562450,73.916619,18.652450,74.006619,2022-03-04,17:30,...,Friday,3,0,Normal,13.787860,2,6,3,108.1,1.0
3,0xc8b6,LUDHRES15DEL02,34.0,4.3,30.899584,75.809346,30.919584,75.829346,2022-02-13,09:20,...,Sunday,2,1,Breakfast,2.930258,1,6,4,146.2,0.0
4,0xdb64,KNPRES14DEL02,24.0,4.7,26.463504,80.372929,26.593504,80.502929,2022-02-14,19:50,...,Monday,2,0,Dinner,19.396618,4,4,3,112.8,1.0


In [3]:
O2A_FEATURES = [
    "Traffic_Score",
    "Peak_Period",
    "Festival",
    "multiple_deliveries",
    "Workload",
    "Rider_Experience",
    "Delivery_person_Ratings"
]

FM_FEATURES = [
    "Trip_Distance_km",
    "Vehicle_Score",
    "Vehicle_condition",
    "Traffic_Score",
    "Restaurant_latitude",
    "Restaurant_longitude",
    "Delivery_person_Ratings"
]

WT_FEATURES = [
    "Weather_Score",
    "Peak_Period",
    "Festival",
    "City",
    "Type_of_order"
]

LM_FEATURES = [
    "Trip_Distance_km",
    "Traffic_Score",
    "Weather_Score",
    "Vehicle_Score",
    "Delivery_location_latitude",
    "Delivery_location_longitude",
    "Rider_Experience"
]

In [4]:
def generate_stage_features(df):

    return {
        "O2A": df[O2A_FEATURES].copy(),
        "FM": df[FM_FEATURES].copy(),
        "WT": df[WT_FEATURES].copy(),
        "LM": df[LM_FEATURES].copy()
    }

stage_features = generate_stage_features(riders)

In [5]:
o2a = stage_features["O2A"]

peak_map = {
    "Breakfast":1,
    "Lunch":2,
    "Dinner":3,
    "Normal":0
}

festival_map = {
    "No":0,
    "Yes":1
}

o2a["Peak_Code"] = o2a["Peak_Period"].map(peak_map)

o2a["Festival_Code"] = o2a["Festival"].map(festival_map)

o2a["O2A_Load_Index"] = (
    o2a["Traffic_Score"] +
    o2a["multiple_deliveries"] +
    o2a["Workload"]
)/3

o2a["O2A_Rider_Index"] = (
    o2a["Rider_Experience"] *
    o2a["Delivery_person_Ratings"]
)

o2a.head()

,Traffic_Score,Peak_Period,Festival,multiple_deliveries,Workload,Rider_Experience,Delivery_person_Ratings,Peak_Code,Festival_Code,O2A_Load_Index,O2A_Rider_Index
0,4,Dinner,No,3.0,3.0,151.2,4.2,3,0,3.333333,635.04
1,3,Lunch,No,1.0,1.0,98.7,4.7,2,0,1.666667,463.89
2,2,Normal,No,1.0,1.0,108.1,4.7,0,0,1.333333,508.07
3,1,Breakfast,No,0.0,0.0,146.2,4.3,1,0,0.333333,628.66
4,4,Dinner,No,1.0,1.0,112.8,4.7,3,0,2.000000,530.16


In [6]:
fm = stage_features["FM"]

fm["FM_Travel_Index"] = (
    fm["Trip_Distance_km"] *
    fm["Traffic_Score"]
)

fm["FM_Vehicle_Index"] = (
    fm["Vehicle_Score"] *
    fm["Vehicle_condition"]
)

fm.head()

,Trip_Distance_km,Vehicle_Score,Vehicle_condition,Traffic_Score,Restaurant_latitude,Restaurant_longitude,Delivery_person_Ratings,FM_Travel_Index,FM_Vehicle_Index
0,10.280582,4,2,4,30.327968,78.046106,4.2,41.122328,8
1,6.242319,4,1,3,10.003064,76.307589,4.7,18.726956,4
2,13.787860,3,1,2,18.562450,73.916619,4.7,27.575720,3
3,2.930258,4,0,1,30.899584,75.809346,4.3,2.930258,0
4,19.396618,3,1,4,26.463504,80.372929,4.7,77.586473,3


In [7]:
wt = stage_features["WT"]

wt["Peak_Code"] = wt["Peak_Period"].map(peak_map)

wt["Festival_Code"] = wt["Festival"].map(festival_map)

city_codes = {
    city:i
    for i,city in enumerate(sorted(riders["City"].unique()))
}

order_codes = {
    order:i
    for i,order in enumerate(sorted(riders["Type_of_order"].unique()))
}

wt["City_Code"] = wt["City"].map(city_codes)

wt["Order_Code"] = wt["Type_of_order"].map(order_codes)

wt["WT_Delay_Index"] = (

    wt["Weather_Score"]

    + wt["Peak_Code"]

    + wt["Festival_Code"]

    + wt["City_Code"]

    + wt["Order_Code"]

)

wt.head()

,Weather_Score,Peak_Period,Festival,City,Type_of_order,Peak_Code,Festival_Code,City_Code,Order_Code,WT_Delay_Index
0,4,Dinner,No,Metropolitian,Snack,3,0,0,3,10
1,5,Lunch,No,Metropolitian,Meal,2,0,0,2,9
2,6,Normal,No,Metropolitian,Drinks,0,0,0,1,7
3,6,Breakfast,No,Metropolitian,Buffet,1,0,0,0,7
4,4,Dinner,No,Metropolitian,Snack,3,0,0,3,10


In [8]:
lm = stage_features["LM"]

lm["LM_Delivery_Index"] = (

    lm["Trip_Distance_km"]

    * lm["Traffic_Score"]

    * lm["Vehicle_Score"]

)

lm.head()

,Trip_Distance_km,Traffic_Score,Weather_Score,Vehicle_Score,Delivery_location_latitude,Delivery_location_longitude,Rider_Experience,LM_Delivery_Index
0,10.280582,4,4,4,30.397968,78.116106,151.2,164.489313
1,6.242319,3,5,4,10.043064,76.347589,98.7,74.907824
2,13.787860,2,6,3,18.652450,74.006619,108.1,82.727161
3,2.930258,1,6,4,30.919584,75.829346,146.2,11.721031
4,19.396618,4,4,3,26.593504,80.502929,112.8,232.759419


In [9]:
fusion = pd.DataFrame()

fusion["O2A_Load_Index"] = o2a["O2A_Load_Index"]

fusion["O2A_Rider_Index"] = o2a["O2A_Rider_Index"]

fusion["FM_Travel_Index"] = fm["FM_Travel_Index"]

fusion["FM_Vehicle_Index"] = fm["FM_Vehicle_Index"]

fusion["WT_Delay_Index"] = wt["WT_Delay_Index"]

fusion["LM_Delivery_Index"] = lm["LM_Delivery_Index"]

In [10]:
global_features = [

    "Order_Hour",

    "Pickup_Hour",

    "Weekend",

    "Month",

    "Delivery_person_Age",

    "Delivery_person_Ratings",

    "Trip_Distance_km",

    "Traffic_Score",

    "Weather_Score",

    "Vehicle_Score",

    "Vehicle_condition",

    "multiple_deliveries",

    "Rider_Experience",

    "City",

    "Festival",

    "Type_of_order"

]

fusion = pd.concat(

    [

        fusion,

        riders[global_features]

    ],

    axis=1

)

fusion.head()

,O2A_Load_Index,O2A_Rider_Index,FM_Travel_Index,FM_Vehicle_Index,WT_Delay_Index,LM_Delivery_Index,Order_Hour,Pickup_Hour,Weekend,Month,...,Trip_Distance_km,Traffic_Score,Weather_Score,Vehicle_Score,Vehicle_condition,multiple_deliveries,Rider_Experience,City,Festival,Type_of_order
0,3.333333,635.04,41.122328,8,10,164.489313,21,22,1,2,...,10.280582,4,4,4,2,3.0,151.2,Metropolitian,No,Snack
1,1.666667,463.89,18.726956,4,9,74.907824,14,15,1,2,...,6.242319,3,5,4,1,1.0,98.7,Metropolitian,No,Meal
2,1.333333,508.07,27.575720,3,7,82.727161,17,17,0,3,...,13.787860,2,6,3,1,1.0,108.1,Metropolitian,No,Drinks
3,0.333333,628.66,2.930258,0,7,11.721031,9,9,1,2,...,2.930258,1,6,4,0,0.0,146.2,Metropolitian,No,Buffet
4,2.000000,530.16,77.586473,3,10,232.759419,19,20,0,2,...,19.396618,4,4,3,1,1.0,112.8,Metropolitian,No,Snack


In [11]:
print(fusion.shape)

fusion.info()

(45493, 22)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45493 entries, 0 to 45492
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   O2A_Load_Index           45493 non-null  float64
 1   O2A_Rider_Index          45493 non-null  float64
 2   FM_Travel_Index          45493 non-null  float64
 3   FM_Vehicle_Index         45493 non-null  int64  
 4   WT_Delay_Index           45493 non-null  int64  
 5   LM_Delivery_Index        45493 non-null  float64
 6   Order_Hour               45493 non-null  int64  
 7   Pickup_Hour              45493 non-null  int64  
 8   Weekend                  45493 non-null  int64  
 9   Month                    45493 non-null  int64  
 10  Delivery_person_Age      45493 non-null  float64
 11  Delivery_person_Ratings  45493 non-null  float64
 12  Trip_Distance_km         45493 non-null  float64
 13  Traffic_Score            45493 non-null  int64  
 14  Weather_Sc

In [12]:
fusion["Time_taken (min)"] = riders["Time_taken (min)"]

fusion.to_csv(

    "../dataset/processed/fusion_features.csv",

    index=False

)

fusion.head()

,O2A_Load_Index,O2A_Rider_Index,FM_Travel_Index,FM_Vehicle_Index,WT_Delay_Index,LM_Delivery_Index,Order_Hour,Pickup_Hour,Weekend,Month,...,Traffic_Score,Weather_Score,Vehicle_Score,Vehicle_condition,multiple_deliveries,Rider_Experience,City,Festival,Type_of_order,Time_taken (min)
0,3.333333,635.04,41.122328,8,10,164.489313,21,22,1,2,...,4,4,4,2,3.0,151.2,Metropolitian,No,Snack,46
1,1.666667,463.89,18.726956,4,9,74.907824,14,15,1,2,...,3,5,4,1,1.0,98.7,Metropolitian,No,Meal,23
2,1.333333,508.07,27.575720,3,7,82.727161,17,17,0,3,...,2,6,3,1,1.0,108.1,Metropolitian,No,Drinks,21
3,0.333333,628.66,2.930258,0,7,11.721031,9,9,1,2,...,1,6,4,0,0.0,146.2,Metropolitian,No,Buffet,20
4,2.000000,530.16,77.586473,3,10,232.759419,19,20,0,2,...,4,4,3,1,1.0,112.8,Metropolitian,No,Snack,41
